# ML2 — Linear Regression Model (Supervised Learning)
- Use Kaggle rental listings dataset (`train.json`) stored in the `data/` folder.
- Build **22 features**: top-20 from the `features` column + `bathrooms` + `bedrooms`.
- Train and evaluate linear models (Linear/Ridge/Lasso/ElasticNet) on train/test splits.
- Implement a **custom Linear Regression (SGD)** and compare it to `sklearn`.
- Implement **custom MinMaxScaler and StandardScaler** and compare to `sklearn`.
- Run a **polynomial degree=10** experiment to observe overfitting and the effect of regularization.
- Report metrics in tables (MAE / RMSE / R²) for train and test.


## Task 1 — Theory Answers 

### 1.a Analytical solution (Normal Equation)
Linear Regression tries to find the best parameters (weights) that make predictions close to real values.  
There are two ways to find these parameters:
- **Iterative training** (like Gradient Descent): improve weights step by step.
- **Analytical solution** (Normal Equation): compute the best weights directly in one step (when the data conditions allow it).  
So, the analytical solution is a direct mathematical way to get the optimal weights without iterative training.

### 1.b Effect of L1 and L2 regularization
Regularization helps prevent overfitting by adding a penalty that discourages complex models.
- **Ridge (L2)**: shrinks all weights smoothly (makes them smaller), reducing overfitting.
- **Lasso (L1)**: can shrink some weights all the way to exactly **zero**, which removes those features from the model.

### 1.c Why L1 selects features
L1 regularization often makes many coefficients exactly **0**.  
If a feature is weak or not useful, L1 tends to remove it by setting its weight to zero.  
That is why Lasso performs **feature selection** automatically.

### 1.d Fitting nonlinear dependencies with linear models
A linear model can still capture nonlinear relationships by transforming the input features first.  
For example, we can create new features such as squares and higher powers (polynomial features) or interaction terms.  
Then we train a linear model on these transformed features.  
The training remains linear in the weights, but the model becomes nonlinear with respect to the original inputs.

## Task 2 — Load data and basic preprocessing

In [1]:
import pandas as pd
import numpy as np

data = pd.read_json("data/train.json")
data.shape, data.columns


((49352, 15),
 Index(['bathrooms', 'bedrooms', 'building_id', 'created', 'description',
        'display_address', 'features', 'latitude', 'listing_id', 'longitude',
        'manager_id', 'photos', 'price', 'street_address', 'interest_level'],
       dtype='object'))

In [2]:
# Quick missing values check
data.isnull().sum()


bathrooms          0
bedrooms           0
building_id        0
created            0
description        0
display_address    0
features           0
latitude           0
listing_id         0
longitude          0
manager_id         0
photos             0
price              0
street_address     0
interest_level     0
dtype: int64

## Building the 22-Feature Dataset

The dataset contains a column called `features`, which is a list of text amenities for each apartment (e.g., `"Elevator"`, `"Doorman"`).  
Machine learning models cannot directly learn from text lists, so we convert the most common amenities into numeric features.

### Steps
1. Collect all amenities from the `features` column across the dataset.
2. Use a frequency counter to find the **Top 20 most common** amenities.
3. For each of these 20 amenities, create a binary column:
   - `1` if the amenity exists in the apartment’s `features` list
   - `0` otherwise
4. Add two numeric features: `bathrooms` and `bedrooms`.

### Final Feature Set
- 20 binary amenity columns (`has_...`)
- + `bathrooms`
- + `bedrooms`

Total = **22 features**, and all models will be trained using this same feature set for a fair comparison.

## Task 3 — Feature engineering from `features` column (Top 20)

In [3]:
from collections import Counter

def clean_token(t: str) -> str:
    # README asks to remove: [, ], ', ", and spaces (we also remove hyphens for consistency)
    return (
        t.replace("[","")
         .replace("]","")
         .replace("'","")
         .replace('"',"")
         .replace(" ","")
         .replace("-","")
    )

# Collect all cleaned feature tokens into one list
all_features = []
for feats in data["features"]:
    all_features.extend([clean_token(x) for x in feats])

# How many unique values?
n_unique = len(set(all_features))
n_unique


1511

In [5]:
# Create 20 binary columns: 1 if the feature exists in the apartment's feature list, else 0
for f in top_20:
    col_name = "has_" + f
    data[col_name] = data["features"].apply(lambda feats: 1 if f in [clean_token(x) for x in feats] else 0)

feature_cols_20 = ["has_" + f for f in top_20]
data[feature_cols_20].head()


,has_Elevator,has_CatsAllowed,has_HardwoodFloors,has_DogsAllowed,has_Doorman,has_Dishwasher,has_NoFee,has_LaundryinBuilding,has_FitnessCenter,has_PreWar,has_LaundryinUnit,has_RoofDeck,has_OutdoorSpace,has_DiningRoom,has_HighSpeedInternet,has_Balcony,has_SwimmingPool,has_LaundryInBuilding,has_NewConstruction,has_Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,0,1,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,0,1,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0


In [4]:
# Top 20 most common cleaned features (as required)
top_20 = [f for f, c in Counter(all_features).most_common(20)]
top_20


['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'PreWar',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [6]:
# Extend feature set with bathrooms and bedrooms -> 22 features total
feature_list = feature_cols_20 + ["bathrooms", "bedrooms"]

df_final = data[feature_list + ["price"]].copy()
df_final.shape, len(feature_list)


((49352, 23), 22)

## Train/Test Split (Why and How)

To evaluate a model fairly, we must test it on data it has never seen before.

### Why do we split the data?
If we train and test on the same data, the model may simply memorize patterns and appear very accurate.  
By using a separate test set, we measure how well the model generalizes to unseen apartments.

### Typical split
We use:
- **80% training data** to learn the model parameters
- **20% test data** to evaluate performance

### Important note: Avoid data leakage
Any preprocessing step that learns from data (e.g., scaling) must be fitted on the training set only,  
then applied to both training and test sets.  
This prevents leaking information from the test set into training.

## Train/Test split

In [7]:
from sklearn.model_selection import train_test_split

X = df_final[feature_list]
y = df_final["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((39481, 22), (9871, 22), (39481,), (9871,))

## Metrics functions

We use standard regression metrics to evaluate models:

- **MAE**: average absolute error between true and predicted prices.
- **MSE**: average squared error (penalizes large errors more).
- **RMSE**: square root of MSE to return error to the original unit (price).
- **R²**: measures how much variance in the target is explained by the model.

We define RMSE as:
RMSE = sqrt(MSE)

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


## Naive Baselines (Mean / Median)

Before training machine learning models, we build simple baseline predictors to understand the minimum performance we should beat.

### Naive Mean baseline
This baseline predicts the same value for every apartment:
- the **mean** of `price` in the training set

It ignores all features and provides a simple reference error level.

### Naive Median baseline
This baseline predicts the same value for every apartment:
- the **median** of `price` in the training set

Median is more robust to extreme prices (outliers), so it can sometimes outperform the mean baseline.

### Why baselines matter
If our trained models cannot beat these naive baselines on the test set,  
then the models are not learning useful information from the features.

In [9]:
mean_price = y_train.mean()
median_price = y_train.median()

y_train_pred_mean = np.full_like(y_train, mean_price, dtype=float)
y_test_pred_mean  = np.full_like(y_test, mean_price, dtype=float)

y_train_pred_median = np.full_like(y_train, median_price, dtype=float)
y_test_pred_median  = np.full_like(y_test, median_price, dtype=float)

naive_mean_metrics = (
    mean_absolute_error(y_train, y_train_pred_mean),
    mean_absolute_error(y_test,  y_test_pred_mean),
    rmse(y_train, y_train_pred_mean),
    rmse(y_test,  y_test_pred_mean),
)

naive_median_metrics = (
    mean_absolute_error(y_train, y_train_pred_median),
    mean_absolute_error(y_test,  y_test_pred_median),
    rmse(y_train, y_train_pred_median),
    rmse(y_test,  y_test_pred_median),
)

naive_mean_metrics, naive_median_metrics


((1443.850253191539,
  1822.4778228263415,
  9861.837860117252,
  45228.003053516004),
 (1324.8664167574277,
  1702.2334110019249,
  9880.194345870706,
  45237.211183778185))

## Training Linear Models (sklearn)

We train and compare four linear models on the same 22-feature dataset:

### 1) Linear Regression
A basic linear model without regularization. It minimizes prediction error directly.

### 2) Ridge Regression (L2)
A linear model with L2 regularization. It shrinks coefficients to reduce overfitting and improves stability.

### 3) Lasso Regression (L1)
A linear model with L1 regularization. It can force some coefficients to become exactly zero, performing feature selection.

### 4) ElasticNet (L1 + L2)
A combination of L1 and L2 regularization. It balances sparsity (L1) and stability (L2), and is useful when features are correlated.

We evaluate each model using MAE, RMSE, and R² on both training and test sets.

In [10]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import pandas as pd

def eval_model(model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred_tr = model.predict(Xtr)
    pred_te = model.predict(Xte)
    return {
        "MAE_train": mean_absolute_error(ytr, pred_tr),
        "MAE_test":  mean_absolute_error(yte, pred_te),
        "RMSE_train": rmse(ytr, pred_tr),
        "RMSE_test":  rmse(yte, pred_te),
        "R2_train": r2_score(ytr, pred_tr),
        "R2_test":  r2_score(yte, pred_te),
    }

models = {
    "linear_sklearn": LinearRegression(),
    "ridge_sklearn": Ridge(alpha=1.0),
    "lasso_sklearn": Lasso(alpha=0.001, max_iter=20000),
    "elasticnet_sklearn": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000),
}

results = []
for name, m in models.items():
    metrics = eval_model(m, X_train, y_train, X_test, y_test)
    results.append({"model": name, **metrics})

results_df = pd.DataFrame(results)
results_df


,model,MAE_train,MAE_test,RMSE_train,RMSE_test,R2_train,R2_test
0,linear_sklearn,993.870610,1368.090376,9712.988808,45186.224164,0.029959,0.001772
1,ridge_sklearn,993.829861,1368.050307,9712.988811,45186.225482,0.029959,0.001772
2,lasso_sklearn,993.867892,1368.087639,9712.988808,45186.224332,0.029959,0.001772
3,elasticnet_sklearn,993.075467,1367.310011,9712.989640,45186.250212,0.029959,0.001771


## Custom Linear Regression (SGD)

To show understanding of how linear regression learns, we implement a custom model using **Stochastic Gradient Descent (SGD)** instead of relying only on `sklearn`.

### Why implement it manually?
`sklearn` can train linear regression in a few lines, but implementing it manually demonstrates how training works internally (updating weights to reduce error).

### What is SGD?
SGD updates the model parameters step by step using individual samples:
1. Initialize weights and bias (e.g., zeros).
2. For each sample:
   - compute prediction
   - compute error
   - update weights and bias with a small step to reduce the error
3. Repeat for multiple passes over the dataset (epochs).

### Deterministic behavior
We use a fixed random seed to make shuffling and training deterministic, so results are reproducible.



In [11]:
X_train_np = X_train.values
y_train_np = y_train.values
X_test_np  = X_test.values
y_test_np  = y_test.values


In [12]:
class MyLinearRegressionSGD:
    def __init__(self, lr=1e-5, epochs=5, seed=42):
        self.lr = lr
        self.epochs = epochs
        self.seed = seed
        self.w = None
        self.b = None

    def fit(self, X, y):
        np.random.seed(self.seed)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.epochs):
            idx = np.arange(n_samples)
            np.random.shuffle(idx)  # deterministic because seed is fixed
            for i in idx:
                x_i = X[i]
                y_i = y[i]
                y_pred = np.dot(x_i, self.w) + self.b
                err = y_pred - y_i
                dw = 2 * err * x_i
                db = 2 * err
                self.w -= self.lr * dw
                self.b -= self.lr * db
        return self

    def predict(self, X):
        return np.dot(X, self.w) + self.b


In [22]:
my_model = MyLinearRegressionSGD(lr=1e-5, epochs=5, seed=42)
my_model.fit(X_train_np, y_train_np)

pred_tr = my_model.predict(X_train_np)
pred_te = my_model.predict(X_test_np)

custom_metrics = {
    "model": "linear_custom_sgd",
    "MAE_train": mean_absolute_error(y_train_np, pred_tr),
    "MAE_test":  mean_absolute_error(y_test_np,  pred_te),
    "RMSE_train": rmse(y_train_np, pred_tr),
    "RMSE_test":  rmse(y_test_np,  pred_te),
    "R2_train": r2_score(y_train_np, pred_tr),
    "R2_test":  r2_score(y_test_np,  pred_te),
}
custom_metrics


{'model': 'linear_custom_sgd',
 'MAE_train': 969.3250870821064,
 'MAE_test': 1343.4187772279552,
 'RMSE_train': 9726.13275134788,
 'RMSE_test': 45191.06630223348,
 'R2_train': 0.027331906433341713,
 'R2_test': 0.0015581128572633718}

### Validation
We compare our custom SGD model metrics (MAE, RMSE, R²) to `sklearn.LinearRegression`.  
Close results indicate that the custom implementation is correct.

## Feature Scaling (MinMaxScaler vs StandardScaler)

Some models are sensitive to the scale of input features, especially:
- gradient-descent based training (SGD)
- regularized linear models (Ridge, Lasso, ElasticNet)

### Why scaling?
Our features have different ranges:
- binary features (0/1)
- numeric features like bathrooms and bedrooms with larger ranges

Scaling makes feature ranges comparable and helps optimization and regularization behave correctly.

### MinMaxScaler
Transforms each feature into the range [0, 1].

### StandardScaler
Transforms each feature to have:
- mean = 0
- standard deviation = 1


In [14]:
class MyMinMaxScaler:
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self

    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_ + 1e-9)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

class MyStandardScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self

    def transform(self, X):
        return (X - self.mean_) / (self.std_ + 1e-9)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)



### to avoid data leakage
We **fit** the scaler on the training set only, then **transform** both training and test sets using the same scaler.

In [15]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# MinMax
mm_custom = MyMinMaxScaler()
X_train_mm = mm_custom.fit_transform(X_train_np)
X_test_mm  = mm_custom.transform(X_test_np)

mm_sk = MinMaxScaler()
X_train_mm_sk = mm_sk.fit_transform(X_train_np)
X_test_mm_sk  = mm_sk.transform(X_test_np)

mm_max_diff = np.max(np.abs(X_train_mm[:5, :5] - X_train_mm_sk[:5, :5]))

# Standard
std_custom = MyStandardScaler()
X_train_std = std_custom.fit_transform(X_train_np)
X_test_std  = std_custom.transform(X_test_np)

std_sk = StandardScaler()
X_train_std_sk = std_sk.fit_transform(X_train_np)
X_test_std_sk  = std_sk.transform(X_test_np)

std_max_diff = np.max(np.abs(X_train_std[:5, :5] - X_train_std_sk[:5, :5]))

mm_max_diff, std_max_diff


(1.000000082740371e-09, 2.3562305440094633e-09)


These values are extremely close to zero, which confirms that the custom scalers match sklearn behavior (minor differences are due to floating-point precision)

### Train and evaluate linear models on MinMax-scaled features.
### We fit each model on the scaled training set, predict on both train and test,
### compute MAE/RMSE/R2, and store the results in a DataFrame for comparison.

In [23]:
models_mm = {
    "linear_mm": LinearRegression(),
    "ridge_mm": Ridge(alpha=1.0),
    "lasso_mm": Lasso(alpha=0.001, max_iter=20000),
    "elasticnet_mm": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000),
}

mm_results = []
for name, m in models_mm.items():
    metrics = eval_model(m, X_train_mm, y_train_np, X_test_mm, y_test_np)
    mm_results.append({"model": name, **metrics})

mm_df = pd.DataFrame(mm_results)
mm_df


,model,MAE_train,MAE_test,RMSE_train,RMSE_test,R2_train,R2_test
0,linear_mm,993.870610,1368.090376,9712.988808,45186.224164,0.029959,0.001772
1,ridge_mm,993.135215,1367.299596,9712.999241,45186.105591,0.029957,0.001777
2,lasso_mm,993.866661,1368.086419,9712.988808,45186.224227,0.029959,0.001772
3,elasticnet_mm,991.032459,1365.762610,9715.424895,45184.973819,0.029472,0.001827


### Same for  Standard-scaled

In [17]:
models_std = {
    "linear_std": LinearRegression(),
    "ridge_std": Ridge(alpha=1.0),
    "lasso_std": Lasso(alpha=0.001, max_iter=20000),
    "elasticnet_std": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000),
}

std_results = []
for name, m in models_std.items():
    metrics = eval_model(m, X_train_std, y_train_np, X_test_std, y_test_np)
    std_results.append({"model": name, **metrics})

std_df = pd.DataFrame(std_results)
std_df


,model,MAE_train,MAE_test,RMSE_train,RMSE_test,R2_train,R2_test
0,linear_std,993.870610,1368.090376,9712.988808,45186.224164,0.029959,0.001772
1,ridge_std,993.862182,1368.082381,9712.988808,45186.224535,0.029959,0.001772
2,lasso_std,993.869506,1368.089297,9712.988808,45186.224243,0.029959,0.001772
3,elasticnet_std,993.704072,1367.932164,9712.988844,45186.231525,0.029959,0.001772


Therefore, scaling does not dramatically change model performance here, although regularized models (especially ElasticNet) show small improvements.

## Polynomial Features (Degree=10)

We expand only `bathrooms` and `bedrooms` using `PolynomialFeatures(degree=10)` to create additional nonlinear terms and interactions.  
This increases the feature count (e.g., from 2 to 65), making the model more flexible but also more likely to overfit or become numerically unstable.  
We fit the transformer on the training set and apply the same transformation to the test set.

In [18]:
from sklearn.preprocessing import PolynomialFeatures

X_poly_base = data[["bathrooms", "bedrooms"]].values
y_poly = data["price"].values

Xtr, Xte, ytr, yte = train_test_split(X_poly_base, y_poly, test_size=0.2, random_state=42)

poly = PolynomialFeatures(degree=10, include_bias=False)
Xtr_poly = poly.fit_transform(Xtr)
Xte_poly = poly.transform(Xte)

Xtr_poly.shape, Xte_poly.shape


((39481, 65), (9871, 65))

### Traing again after poly ( to observe overfitting and the effect of regularization)

In [19]:
poly_models = {
    "linear_poly": LinearRegression(),
    "ridge_poly": Ridge(alpha=1.0),
    "lasso_poly": Lasso(alpha=0.001, max_iter=20000),
    "elasticnet_poly": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000),
}

poly_results = []
for name, m in poly_models.items():
    metrics = eval_model(m, Xtr_poly, ytr, Xte_poly, yte)
    poly_results.append({"model": name, **metrics})

poly_df = pd.DataFrame(poly_results)
poly_df


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=4.19292e-21): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.858e+12, tolerance: 3.840e+08
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.858e+12, tolerance: 3.840e+08
  m

,model,MAE_train,MAE_test,RMSE_train,RMSE_test,R2_train,R2_test
0,linear_poly,950.782690,1328.366674,9693.898695,45196.411060,0.033768,0.001322
1,ridge_poly,949.687719,1327.181491,9694.087948,45196.472755,0.033731,0.001319
2,lasso_poly,958.948583,1335.402338,9700.742399,45197.137302,0.032404,0.001290
3,elasticnet_poly,958.623700,1334.998727,9700.801374,45197.096938,0.032392,0.001292


Using polynomial features (degree=10) slightly improved MAE on the test set (around 1327–1335 vs ~1368 before).  
However, the R² score on the test set remains very low, which suggests that the current feature set and the presence of extreme price values limit the model’s ability to explain variance in `price`.


## Final Metrics Tables

In [20]:
result_MAE = pd.DataFrame(columns=["model", "train", "test"])
result_RMSE = pd.DataFrame(columns=["model", "train", "test"])
result_R2 = pd.DataFrame(columns=["model", "train", "test"])

def add_to_tables(model_name, mae_tr, mae_te, rmse_tr, rmse_te, r2_tr, r2_te):
    result_MAE.loc[len(result_MAE)] = [model_name, mae_tr, mae_te]
    result_RMSE.loc[len(result_RMSE)] = [model_name, rmse_tr, rmse_te]
    result_R2.loc[len(result_R2)] = [model_name, r2_tr, r2_te]

# sklearn base
for _, row in results_df.iterrows():
    add_to_tables(row["model"], row["MAE_train"], row["MAE_test"], row["RMSE_train"], row["RMSE_test"], row["R2_train"], row["R2_test"])

# custom SGD
add_to_tables(custom_metrics["model"], custom_metrics["MAE_train"], custom_metrics["MAE_test"],
              custom_metrics["RMSE_train"], custom_metrics["RMSE_test"],
              custom_metrics["R2_train"], custom_metrics["R2_test"])

# naive (R2 not meaningful here -> NaN)
add_to_tables("naive_mean", naive_mean_metrics[0], naive_mean_metrics[1], naive_mean_metrics[2], naive_mean_metrics[3], np.nan, np.nan)
add_to_tables("naive_median", naive_median_metrics[0], naive_median_metrics[1], naive_median_metrics[2], naive_median_metrics[3], np.nan, np.nan)

# scaled results
for _, row in mm_df.iterrows():
    add_to_tables(row["model"], row["MAE_train"], row["MAE_test"], row["RMSE_train"], row["RMSE_test"], row["R2_train"], row["R2_test"])
for _, row in std_df.iterrows():
    add_to_tables(row["model"], row["MAE_train"], row["MAE_test"], row["RMSE_train"], row["RMSE_test"], row["R2_train"], row["R2_test"])

# polynomial results
for _, row in poly_df.iterrows():
    add_to_tables(row["model"], row["MAE_train"], row["MAE_test"], row["RMSE_train"], row["RMSE_test"], row["R2_train"], row["R2_test"])

result_MAE.sort_values("test").head(10), result_RMSE.sort_values("test").head(10), result_R2.sort_values("test", ascending=False).head(10)


(                 model       train         test
 16          ridge_poly  949.687719  1327.181491
 15         linear_poly  950.782690  1328.366674
 18     elasticnet_poly  958.623700  1334.998727
 17          lasso_poly  958.948583  1335.402338
 4    linear_custom_sgd  969.325087  1343.418777
 10       elasticnet_mm  991.032459  1365.762610
 8             ridge_mm  993.135215  1367.299596
 3   elasticnet_sklearn  993.075467  1367.310011
 14      elasticnet_std  993.704072  1367.932164
 1        ridge_sklearn  993.829861  1368.050307,
              model        train          test
 10   elasticnet_mm  9715.424895  45184.973819
 8         ridge_mm  9712.999241  45186.105591
 0   linear_sklearn  9712.988808  45186.224164
 11      linear_std  9712.988808  45186.224164
 7        linear_mm  9712.988808  45186.224164
 9         lasso_mm  9712.988808  45186.224227
 13       lasso_std  9712.988808  45186.224243
 2    lasso_sklearn  9712.988808  45186.224332
 12       ridge_std  9712.988808  451

## Conclusion

- **Best model (lowest MAE on test):** `ridge_poly`  
  It achieved the lowest test MAE (~1327), which means it produced the most accurate average price predictions among all tested models.

- **Most stable model (consistent train/test behavior):** `elasticnet_mm`  
  It showed the best test R² among the scaled linear models and maintained stable performance without the extra complexity of polynomial expansion.

Note: RMSE values remain very large across models due to extreme price outliers. This is expected because RMSE heavily penalizes large errors.